## 准备数据

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [3]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [4]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([28*28, 100], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 28*28])
        h1 = x @ self.W1 + self.b1
        h1_relu = tf.nn.relu(h1)
        logits = h1_relu @ self.W2 + self.b2
        
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [5]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [6]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.3521204 ; accuracy 0.16133334
epoch 1 : loss 2.336897 ; accuracy 0.16391666
epoch 2 : loss 2.322512 ; accuracy 0.16603333
epoch 3 : loss 2.3088713 ; accuracy 0.16913334
epoch 4 : loss 2.2958963 ; accuracy 0.17213333
epoch 5 : loss 2.2835171 ; accuracy 0.17483333
epoch 6 : loss 2.2716734 ; accuracy 0.17788333
epoch 7 : loss 2.2603145 ; accuracy 0.18035
epoch 8 : loss 2.2493963 ; accuracy 0.18336667
epoch 9 : loss 2.2388766 ; accuracy 0.18611667
epoch 10 : loss 2.2287204 ; accuracy 0.1895
epoch 11 : loss 2.2188952 ; accuracy 0.1927
epoch 12 : loss 2.2093744 ; accuracy 0.19596666
epoch 13 : loss 2.2001324 ; accuracy 0.19906667
epoch 14 : loss 2.1911447 ; accuracy 0.20185
epoch 15 : loss 2.1823924 ; accuracy 0.20511666
epoch 16 : loss 2.173856 ; accuracy 0.2076
epoch 17 : loss 2.1655207 ; accuracy 0.21115
epoch 18 : loss 2.157373 ; accuracy 0.214
epoch 19 : loss 2.1493971 ; accuracy 0.218
epoch 20 : loss 2.141581 ; accuracy 0.22248334
epoch 21 : loss 2.1339145 ; accuracy 0